In [0]:
from pyspark.sql.functions import(col, concat, lit, to_timestamp, to_date,try_to_timestamp, concat_ws,trim, upper, lower )

In [0]:
%run "../includes/common_functions"

In [0]:
races_df = spark.read.table("f1.bronze.races")

In [0]:
races_renamed_df = races_df\
    .withColumnRenamed("raceId", "race_id")\
    .withColumnRenamed("circuitId", "circuit_id")\
    .withColumnRenamed("name","race_name")

In [0]:
races_renamed_df = races_renamed_df \
    .withColumn("race_name", trim(col("race_name"))) \
    .withColumn("url", trim(col("url")))

In [0]:


races_rename_date_df = races_renamed_df.withColumn(
    "race_timestamp",
    try_to_timestamp(
        concat(col("date"), lit(" "), col("time")),
        lit("yyyy-MM-dd HH:mm:ss")
    )
)

In [0]:
races_clean_df = races_rename_date_df \
    .filter(col("race_id").isNotNull()) \
    .dropDuplicates(["race_id"])

In [0]:
races_final_df = add_ingestion_date(races_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
races_final_df.createOrReplaceTempView("races_update")


In [0]:
if spark.catalog.tableExists("f1.silver.races"):
    spark.sql("""
        
        MERGE INTO f1.silver.races tgt
        USING races_update src
        ON tgt.race_id = src.race_id

        WHEN MATCHED THEN
        UPDATE SET *

        WHEN NOT MATCHED THEN
        INSERT *

        """)
else: 
        races_final_df.wirte\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable("f1.silver.races")

In [0]:
%sql
SELECT * FROM f1.silver.races;